In [1]:
import pandas as pd
import bs4 as bs
import re
from group import RateGroup, RateAgreement, RateSteps
import json

In [2]:
classDf = pd.read_pickle('classDf.pkl')
idContentDf = pd.read_pickle('idContentDf.pkl')

In [3]:
id_3 = classDf.loc[classDf['ids'] == '3']
ac = idContentDf.loc[idContentDf['ids'] == '3', 'htmlContent'].values[0]
soup = bs.BeautifulSoup(ac,'lxml')

rateTables = soup.select('h3:has(span[id^="rates"]) ~ table')
rates = []
for rateCat in rateTables:
    steps = rateCat.select('thead th')
    rateStpes = rateCat.select('tbody tr')
    rateAgreements = []
    for idx,agreement in enumerate(rateStpes):
        rateStepsList = []
        for idx2,amount in enumerate(agreement.findAll('td')):
            step = re.search('Step.[0-9]*', steps[idx2+1].getText())
            if amount.find(' to ') == -1:
                newAmts = [amountgetText().replace(',', '')]
            else:
                newAmts = amount.getText().replace(',', '').split(' to ')
            rateStep = RateSteps(
                int(re.search('[0-9]',step[0])[0]),
                [int(newAmt) for newAmt in newAmts]
            )
            rateStepsList.append(rateStep)
        rateAgreement = RateAgreement(
            agreement.find('time').get('datetime'),
            rateStepsList
        )
        rateAgreements.append(rateAgreement)
    rateGrp = RateGroup(
        rateCat.find('caption').getText().split(":")[0],
        rateCat.find('caption').getText().split(":")[0].rsplit('-',1)[0].strip(),
        rateCat.find('caption').getText().split(":")[0].rsplit('-',1)[1].strip(),
        rateAgreements
    )
    rates.append(rateGrp)

TypeError: Object of type RateGroup is not JSON serializable